# Smart Postal ML System Training
## 2-Model Architecture for Intelligent Mail Processing

**Objective**: Train ML models to:
1. Classify mail priority (urgent vs regular)
2. Optimize delivery routes with dynamic rerouting

In [1]:
# Install required packages
import subprocess
import sys

def install_package(package):
    """Install package if not already installed"""
    try:
        __import__(package.split('[')[0].replace('-', '_'))
        print(f"✅ {package} already installed")
    except ImportError:
        print(f"📦 Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        print(f"✅ {package} installed successfully")

packages = ['pandas', 'numpy', 'scikit-learn', 'xgboost', 'matplotlib', 'seaborn']
for pkg in packages:
    install_package(pkg)

print("\n🎉 All packages ready!")

✅ pandas already installed
✅ numpy already installed
📦 Installing scikit-learn...
✅ scikit-learn installed successfully
✅ xgboost already installed
✅ matplotlib already installed
✅ seaborn already installed

🎉 All packages ready!


In [2]:
# Standard library imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
import pickle
import random
from typing import Dict, List, Tuple
from math import radians, sin, cos, sqrt, atan2

# ML imports
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score, roc_curve
)
from sklearn.utils.class_weight import compute_class_weight
import xgboost as xgb

# Configuration
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
warnings.filterwarnings('ignore')

# Plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print("✅ Libraries imported successfully")
print(f"📌 Random seed: {RANDOM_SEED}")
print(f"🐍 Python version: {sys.version.split()[0]}")

✅ Libraries imported successfully
📌 Random seed: 42
🐍 Python version: 3.12.5


In [5]:
class PriorityClassificationModel:
    """
    MODEL 1: Priority Classification using XGBoost
    
    Classifies mail items as urgent or regular based on attributes
    using sophisticated feature engineering and class imbalance handling.
    """
    
    def __init__(self, random_state: int = 42):
        """Initialize the priority classification model"""
        self.random_state = random_state
        self.model = None
        self.encoders = {}
        self.scaler = StandardScaler()
        self.feature_names = []
        self.is_trained = False
        
        # Define feature spaces (Sri Lankan postal context)
        self.MAIL_TYPES = [
            'Court Notice', 'Legal Document', 'Registered Letter', 'Speed Post',
            'Express Mail', 'Tax Document', 'Government Letter', 'Bank Document',
            'Medical Report', 'Insurance Document', 'Certificate', 'Parcel',
            'Standard Letter', 'Magazine', 'Bill', 'Advertisement'
        ]
        
        self.SENDER_TYPES = [
            'Court', 'Law Firm', 'Government Office', 'Tax Office', 'Bank',
            'Hospital', 'Insurance Company', 'Educational Institute',
            'Business', 'Individual', 'NGO'
        ]
        
        self.RECIPIENT_TYPES = [
            'Individual', 'Business', 'Government Office', 'Law Firm',
            'Educational Institute', 'Hospital', 'Bank', 'Insurance Company'
        ]
        
        self.TIME_SLOTS = ['08:00', '09:30', '11:00', '13:00', '14:30', '16:00']
        self.DAYS_OF_WEEK = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
    
    def generate_training_data(self, n_samples: int = 5000) -> pd.DataFrame:
        """
        Generate synthetic training data with realistic patterns
        
        Business Rules for Priority Assignment:
        ---------------------------------------
        1. Legal/Court documents → High priority (Score: +4)
        2. Government + Tax documents → High priority (Score: +3)
        3. Registered/Express mail → High priority (Score: +4)
        4. Medical reports from hospitals → Medium priority (Score: +2)
        5. Early morning delivery → Boost priority (Score: +1)
        6. Monday/Tuesday delivery → Slight boost (Score: +1)
        7. Priority sender + Individual recipient → Boost (Score: +2)
        
        Threshold: Score ≥ 5 → URGENT
        
        Parameters:
        -----------
        n_samples : int
            Number of training samples to generate
            
        Returns:
        --------
        pd.DataFrame : Training dataset with labels
        """
        
        print(f"\n📊 Generating {n_samples:,} training samples...")
        
        records = []
        
        for i in range(n_samples):
            # Random selection
            mail_type = random.choice(self.MAIL_TYPES)
            sender_type = random.choice(self.SENDER_TYPES)
            recipient_type = random.choice(self.RECIPIENT_TYPES)
            time_received = random.choice(self.TIME_SLOTS)
            day_of_week = random.choice(self.DAYS_OF_WEEK)
            
            # Urgency scoring logic (0-10 scale)
            urgency_score = 0
            
            # Rule 1: High priority mail types
            if mail_type in ['Court Notice', 'Legal Document', 'Registered Letter', 
                           'Speed Post', 'Express Mail', 'Tax Document', 'Certificate']:
                urgency_score += 4
            
            # Rule 2: High priority senders
            if sender_type in ['Court', 'Law Firm', 'Government Office', 'Tax Office']:
                urgency_score += 3
            
            # Rule 3: Specific combinations
            if mail_type == 'Medical Report' and sender_type == 'Hospital':
                urgency_score += 2
            if mail_type in ['Bank Document', 'Insurance Document'] and recipient_type in ['Business', 'Individual']:
                urgency_score += 1
            
            # Rule 4: Time sensitivity
            if time_received in ['08:00', '09:30']:
                urgency_score += 1
            
            # Rule 5: Day sensitivity
            if day_of_week in ['Monday', 'Tuesday']:
                urgency_score += 1
            
            # Rule 6: Critical combinations
            if sender_type in ['Court', 'Law Firm'] and recipient_type == 'Individual':
                urgency_score += 2
            
            # Rule 7: Express combinations
            if mail_type in ['Speed Post', 'Express Mail'] and time_received in ['08:00', '09:30']:
                urgency_score += 1
            
            # Determine urgency (threshold: 5)
            is_urgent = urgency_score >= 5
            
            # Add 2% noise for realism
            if random.random() < 0.02:
                is_urgent = not is_urgent
            
            records.append({
                'mail_id': f'MAIL{i+1:06d}',
                'mail_type': mail_type,
                'sender_type': sender_type,
                'recipient_type': recipient_type,
                'time_received': time_received,
                'day_of_week': day_of_week,
                'urgency_score': urgency_score,
                'priority': 'urgent' if is_urgent else 'regular'
            })
        
        df = pd.DataFrame(records)
        
        # Statistics
        class_counts = df['priority'].value_counts()
        print(f"✅ Dataset created successfully")
        print(f"\n   Class distribution:")
        for label, count in class_counts.items():
            print(f"   • {label}: {count:,} ({count/len(df)*100:.1f}%)")
        
        return df
    
    def preprocess_features(self, df: pd.DataFrame, fit: bool = True) -> np.ndarray:
        """
        Comprehensive feature engineering pipeline
        
        Steps:
        ------
        1. Encode categorical features using LabelEncoder
        2. Create temporal features (time category)
        3. Create binary indicator features
        4. Create interaction features
        5. Scale all features using StandardScaler
        
        Parameters:
        -----------
        df : pd.DataFrame
            Input dataframe with raw features
        fit : bool
            If True, fit encoders and scaler; if False, transform only
            
        Returns:
        --------
        np.ndarray : Processed feature matrix
        """
        
        df = df.copy()
        
        # 1. Encode categorical features
        categorical_features = ['mail_type', 'sender_type', 'recipient_type', 
                               'time_received', 'day_of_week']
        
        for feature in categorical_features:
            if fit:
                self.encoders[feature] = LabelEncoder()
                df[f'{feature}_encoded'] = self.encoders[feature].fit_transform(df[feature])
            else:
                df[f'{feature}_encoded'] = self.encoders[feature].transform(df[feature])
        
        # 2. Temporal features
        time_to_category = {
            '08:00': 0, '09:30': 0,  # Early morning
            '11:00': 1, '13:00': 1,  # Mid-day
            '14:30': 2, '16:00': 2   # Afternoon
        }
        df['time_category'] = df['time_received'].map(time_to_category)
        
        # 3. Binary indicators
        df['is_priority_sender'] = df['sender_type'].isin(
            ['Court', 'Law Firm', 'Government Office', 'Tax Office']
        ).astype(int)
        
        df['is_priority_mail'] = df['mail_type'].isin(
            ['Court Notice', 'Legal Document', 'Registered Letter', 
             'Speed Post', 'Express Mail', 'Tax Document', 'Certificate']
        ).astype(int)
        
        df['is_early_week'] = df['day_of_week'].isin(['Monday', 'Tuesday']).astype(int)
        df['is_morning'] = df['time_received'].isin(['08:00', '09:30']).astype(int)
        
        # 4. Interaction features
        df['priority_sender_mail'] = df['is_priority_sender'] * df['is_priority_mail']
        df['morning_priority'] = df['is_morning'] * df['is_priority_mail']
        df['early_week_priority'] = df['is_early_week'] * df['is_priority_mail']
        df['morning_early_week'] = df['is_morning'] * df['is_early_week']
        
        # 5. Select feature columns
        self.feature_names = [
            'mail_type_encoded', 'sender_type_encoded', 'recipient_type_encoded',
            'time_received_encoded', 'day_of_week_encoded', 'time_category',
            'is_priority_sender', 'is_priority_mail', 'is_early_week', 'is_morning',
            'priority_sender_mail', 'morning_priority', 'early_week_priority',
            'morning_early_week'
        ]
        
        X = df[self.feature_names].values
        
        # 6. Scale features
        if fit:
            X = self.scaler.fit_transform(X)
        else:
            X = self.scaler.transform(X)
        
        return X
    
    def train(self, df: pd.DataFrame, test_size: float = 0.2, 
             tune_hyperparameters: bool = False) -> Dict:
        """
        Train the priority classification model
        
        Parameters:
        -----------
        df : pd.DataFrame
            Training data with labels
        test_size : float
            Proportion of data for testing (default: 0.2)
        tune_hyperparameters : bool
            If True, perform GridSearchCV for hyperparameter tuning
            
        Returns:
        --------
        dict : Training results including metrics and evaluation data
        """
        
        print(f"\n🔧 Training Priority Classification Model...")
        print(f"   Training samples: {int(len(df) * (1-test_size)):,}")
        print(f"   Test samples: {int(len(df) * test_size):,}")
        
        # Preprocess features
        X = self.preprocess_features(df, fit=True)
        
        # Encode target
        le_target = LabelEncoder()
        y = le_target.fit_transform(df['priority'])
        self.encoders['target'] = le_target
        
        # Split data with stratification
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=self.random_state, stratify=y
        )
        
        # Calculate class weights
        class_weights = compute_class_weight(
            'balanced', 
            classes=np.unique(y_train), 
            y=y_train
        )
        scale_pos_weight = class_weights[1] / class_weights[0]
        
        print(f"   Class weight ratio: {scale_pos_weight:.2f} (favoring urgent class)")
        
        # Model training
        if tune_hyperparameters:
            print(f"\n   🔍 Tuning hyperparameters with GridSearchCV...")
            
            param_grid = {
                'n_estimators': [200, 300, 400],
                'max_depth': [6, 8, 10],
                'learning_rate': [0.05, 0.1, 0.15],
                'subsample': [0.8, 0.9],
                'colsample_bytree': [0.8, 0.9]
            }
            
            xgb_base = xgb.XGBClassifier(
                scale_pos_weight=scale_pos_weight,
                random_state=self.random_state,
                eval_metric='logloss',
                use_label_encoder=False
            )
            
            grid_search = GridSearchCV(
                xgb_base,
                param_grid,
                cv=5,
                scoring='recall',
                n_jobs=-1,
                verbose=1
            )
            
            grid_search.fit(X_train, y_train)
            self.model = grid_search.best_estimator_
            
            print(f"\n   ✅ Best parameters:")
            for param, value in grid_search.best_params_.items():
                print(f"      • {param}: {value}")
        
        else:
            # Use optimized default parameters
            self.model = xgb.XGBClassifier(
                n_estimators=300,
                max_depth=8,
                learning_rate=0.1,
                subsample=0.9,
                colsample_bytree=0.9,
                scale_pos_weight=scale_pos_weight,
                random_state=self.random_state,
                eval_metric='logloss',
                use_label_encoder=False
            )
            
            print(f"\n   🎯 Training with optimized parameters...")
            self.model.fit(X_train, y_train)
        
        self.is_trained = True
        
        # Predictions
        y_pred = self.model.predict(X_test)
        y_pred_proba = self.model.predict_proba(X_test)
        
        # Calculate metrics
        metrics = {
            'accuracy': accuracy_score(y_test, y_pred),
            'precision': precision_score(y_test, y_pred),
            'recall': recall_score(y_test, y_pred),
            'f1_score': f1_score(y_test, y_pred),
            'roc_auc': roc_auc_score(y_test, y_pred_proba[:, 1])
        }
        
        # Cross-validation
        cv_scores_recall = cross_val_score(
            self.model, X_train, y_train, cv=5, scoring='recall'
        )
        cv_scores_precision = cross_val_score(
            self.model, X_train, y_train, cv=5, scoring='precision'
        )
        
        metrics['cv_recall_mean'] = cv_scores_recall.mean()
        metrics['cv_recall_std'] = cv_scores_recall.std()
        metrics['cv_precision_mean'] = cv_scores_precision.mean()
        metrics['cv_precision_std'] = cv_scores_precision.std()
        
        # Confusion matrix
        cm = confusion_matrix(y_test, y_pred)
        
        # Feature importance
        feature_importance = pd.DataFrame({
            'feature': self.feature_names,
            'importance': self.model.feature_importances_
        }).sort_values('importance', ascending=False)
        
        # Print results
        print(f"\n{'=' * 70}")
        print("📊 MODEL 1: PERFORMANCE METRICS")
        print(f"{'=' * 70}")
        print(f"   Accuracy:  {metrics['accuracy']:.4f} ({metrics['accuracy']*100:.2f}%)")
        print(f"   Precision: {metrics['precision']:.4f} ({metrics['precision']*100:.2f}%)")
        
        recall_status = "✅ TARGET MET" if metrics['recall'] >= 0.95 else "⚠️  Below target"
        print(f"   Recall:    {metrics['recall']:.4f} ({metrics['recall']*100:.2f}%) {recall_status}")
        print(f"   F1-Score:  {metrics['f1_score']:.4f}")
        print(f"   ROC-AUC:   {metrics['roc_auc']:.4f}")
        
        print(f"\n   Cross-Validation (5-fold):")
        print(f"   Recall:    {metrics['cv_recall_mean']:.4f} (±{metrics['cv_recall_std']:.4f})")
        print(f"   Precision: {metrics['cv_precision_mean']:.4f} (±{metrics['cv_precision_std']:.4f})")
        
        print(f"\n   Confusion Matrix:")
        print(f"                    Predicted")
        print(f"                Regular  Urgent")
        print(f"   Actual Regular  {cm[0,0]:5d}   {cm[0,1]:5d}")
        print(f"          Urgent   {cm[1,0]:5d}   {cm[1,1]:5d}")
        
        print(f"\n   Top 5 Important Features:")
        for idx, row in feature_importance.head(5).iterrows():
            print(f"   • {row['feature']:25s}: {row['importance']:.4f}")
        
        return {
            'model': self.model,
            'metrics': metrics,
            'confusion_matrix': cm,
            'feature_importance': feature_importance,
            'X_test': X_test,
            'y_test': y_test,
            'y_pred': y_pred,
            'y_pred_proba': y_pred_proba
        }
    
    def predict(self, mail_data: Dict) -> Dict:
        """
        Predict priority for a single mail item
        
        Parameters:
        -----------
        mail_data : dict
            Dictionary containing mail attributes
            
        Returns:
        --------
        dict : Prediction results with confidence scores
        """
        
        if not self.is_trained:
            raise ValueError("Model not trained. Call train() first.")
        
        df = pd.DataFrame([mail_data])
        X = self.preprocess_features(df, fit=False)
        
        prediction = self.model.predict(X)[0]
        probabilities = self.model.predict_proba(X)[0]
        
        priority_label = self.encoders['target'].inverse_transform([prediction])[0]
        
        return {
            'priority': priority_label,
            'confidence': float(probabilities.max()),
            'probability_regular': float(probabilities[0]),
            'probability_urgent': float(probabilities[1]),
            'prediction_class': int(prediction)
        }
    
    def save_model(self, filepath: str):
        """Save trained model to disk"""
        model_data = {
            'model': self.model,
            'encoders': self.encoders,
            'scaler': self.scaler,
            'feature_names': self.feature_names,
            'random_state': self.random_state
        }
        
        with open(filepath, 'wb') as f:
            pickle.dump(model_data, f)
        
        print(f"✅ Model 1 saved to {filepath}")

In [6]:
# Initialize the model
print("=" * 70)
print("MODEL 1: PRIORITY CLASSIFICATION - DATA GENERATION")
print("=" * 70)

priority_model = PriorityClassificationModel(random_state=RANDOM_SEED)

# Generate training data
df_priority = priority_model.generate_training_data(n_samples=5000)

# Display sample data
print("\n📊 Sample Data:")
display(df_priority.head(10))

print("\n📈 Dataset Statistics:")
print(df_priority.describe())

MODEL 1: PRIORITY CLASSIFICATION - DATA GENERATION

📊 Generating 5,000 training samples...
✅ Dataset created successfully

   Class distribution:
   • regular: 3,151 (63.0%)
   • urgent: 1,849 (37.0%)

📊 Sample Data:


,mail_id,mail_type,sender_type,recipient_type,time_received,day_of_week,urgency_score,priority
0,MAIL000001,Speed Post,Court,Educational Institute,09:30,Tuesday,10,urgent
1,MAIL000002,Speed Post,NGO,Business,14:30,Thursday,4,regular
2,MAIL000003,Registered Letter,Tax Office,Law Firm,14:30,Friday,7,urgent
3,MAIL000004,Government Letter,NGO,Bank,09:30,Thursday,1,regular
4,MAIL000005,Court Notice,Government Office,Bank,11:00,Wednesday,7,urgent
5,MAIL000006,Certificate,Law Firm,Business,13:00,Monday,8,urgent
6,MAIL000007,Parcel,Individual,Educational Institute,08:00,Saturday,1,regular
7,MAIL000008,Speed Post,Insurance Company,Business,14:30,Wednesday,4,regular
8,MAIL000009,Parcel,Individual,Law Firm,16:00,Monday,1,regular
9,MAIL000010,Bank Document,Bank,Business,09:30,Monday,3,regular



📈 Dataset Statistics:
       urgency_score
count    5000.000000
mean        3.720200
std         2.656864
min         0.000000
25%         1.000000
50%         4.000000
75%         5.000000
max        12.000000


In [7]:
print("=" * 70)
print("MODEL 1: TRAINING")
print("=" * 70)

# Train the model
priority_results = priority_model.train(
    df_priority, 
    test_size=0.2,           # 20% for testing
    tune_hyperparameters=False  # Set True for hyperparameter tuning (slower)
)

print("\n✅ Model 1 training complete!")

MODEL 1: TRAINING

🔧 Training Priority Classification Model...
   Training samples: 4,000
   Test samples: 1,000
   Class weight ratio: 1.70 (favoring urgent class)

   🎯 Training with optimized parameters...

📊 MODEL 1: PERFORMANCE METRICS
   Accuracy:  0.9730 (97.30%)
   Precision: 0.9673 (96.73%)
   Recall:    0.9595 (95.95%) ✅ TARGET MET
   F1-Score:  0.9634
   ROC-AUC:   0.9744

   Cross-Validation (5-fold):
   Recall:    0.9527 (±0.0156)
   Precision: 0.9698 (±0.0064)

   Confusion Matrix:
                    Predicted
                Regular  Urgent
   Actual Regular    618      12
          Urgent      15     355

   Top 5 Important Features:
   • is_priority_mail         : 0.4531
   • early_week_priority      : 0.1545
   • priority_sender_mail     : 0.0753
   • morning_priority         : 0.0685
   • morning_early_week       : 0.0683

✅ Model 1 training complete!


In [8]:
print("=" * 70)
print("MODEL 1: PREDICTION TESTING")
print("=" * 70)

test_cases = [
    {
        'name': '🚨 Urgent: Court Notice',
        'mail_type': 'Court Notice',
        'sender_type': 'Court',
        'recipient_type': 'Individual',
        'time_received': '08:00',
        'day_of_week': 'Monday'
    },
    {
        'name': '📬 Regular: Advertisement',
        'mail_type': 'Advertisement',
        'sender_type': 'Business',
        'recipient_type': 'Individual',
        'time_received': '14:30',
        'day_of_week': 'Friday'
    },
    {
        'name': '⚠️ Important: Tax Document',
        'mail_type': 'Tax Document',
        'sender_type': 'Tax Office',
        'recipient_type': 'Business',
        'time_received': '09:30',
        'day_of_week': 'Tuesday'
    }
]

for i, test_case in enumerate(test_cases, 1):
    name = test_case.pop('name')
    result = priority_model.predict(test_case)
    
    print(f"\n{'='*60}")
    print(f"Test Case {i}: {name}")
    print(f"{'='*60}")
    print(f"📧 Mail Type: {test_case['mail_type']}")
    print(f"👤 Sender: {test_case['sender_type']}")
    print(f"🎯 Prediction: {result['priority'].upper()}")
    print(f"📊 Confidence: {result['confidence']:.1%}")
    print(f"📈 Probabilities:")
    print(f"   • Regular: {result['probability_regular']:.1%}")
    print(f"   • Urgent:  {result['probability_urgent']:.1%}")

MODEL 1: PREDICTION TESTING

Test Case 1: 🚨 Urgent: Court Notice
📧 Mail Type: Court Notice
👤 Sender: Court
🎯 Prediction: URGENT
📊 Confidence: 100.0%
📈 Probabilities:
   • Regular: 0.0%
   • Urgent:  100.0%

Test Case 2: 📬 Regular: Advertisement
📧 Mail Type: Advertisement
👤 Sender: Business
🎯 Prediction: REGULAR
📊 Confidence: 99.8%
📈 Probabilities:
   • Regular: 99.8%
   • Urgent:  0.2%

Test Case 3: ⚠️ Important: Tax Document
📧 Mail Type: Tax Document
👤 Sender: Tax Office
🎯 Prediction: URGENT
📊 Confidence: 99.9%
📈 Probabilities:
   • Regular: 0.1%
   • Urgent:  99.9%
